In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 24. 디코딩 전략 — 텍스트 생성의 예술

> **학습 목표**
> - Greedy, Beam Search, Top-k, Top-p, Temperature의 수학을 이해한다
> - 각 전략이 생성 품질·다양성에 미치는 영향을 실험한다
> - Speculative Decoding의 원리를 설명한다

## 24.1 디코딩 문제 정의

LLM은 각 위치에서 어휘에 대한 확률 분포 $P(w_t | w_{<t})$ 출력.
이를 실제 토큰으로 변환하는 전략 = 디코딩.

트레이드오프: **결정성(품질) vs 다양성(창의성)**

## 24.2 Greedy Decoding

매 스텝 가장 확률 높은 토큰 선택:
$$w_t = \arg\max_w P(w | w_{<t})$$

- 빠르고 결정적
- 단조로운 텍스트, 반복 위험


In [ ]:
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

# 가짜 로짓 시퀀스 (10 스텝, 어휘 20)
torch.manual_seed(0)
vocab_size = 20
n_steps = 10
logits_seq = torch.randn(n_steps, vocab_size)

def softmax(x, tau=1.0):
    x = x / tau
    x = x - x.max()
    e = np.exp(x)
    return e / e.sum()

def greedy_decode(logits_seq):
    """Greedy: 매 Step argmax."""
    tokens = []
    for logits in logits_seq:
        tokens.append(logits.argmax().item())
    return tokens

greedy_tokens = greedy_decode(logits_seq)
print(f"Greedy: {greedy_tokens}")


## 24.3 Temperature Sampling

$$P_\tau(w) = \frac{\exp(\log P(w) / \tau)}{\sum_{w'} \exp(\log P(w') / \tau)}$$

- $\tau \to 0$: Greedy에 수렴 (날카로운 분포)
- $\tau = 1$: 원래 소프트맥스
- $\tau \to \infty$: 균일 분포 (완전 랜덤)


In [ ]:
# Temperature 효과
logits = torch.randn(20)  # 20 토큰에 대한 로짓
print(f"로짓: {logits.numpy().round(2)}")
print()
for tau in [0.3, 1.0, 2.0, 5.0]:
    p = softmax(logits.numpy(), tau=tau)
    print(f"tau={tau}: {p.round(4)}  entropy={-np.sum(p*np.log(p+1e-12)):.3f}")
print("\n=> tau 작을수록 날카로운 Distribution(결정적), 클수록 평평한(랜덤).")


## 24.4 Top-k Sampling

가장 확률 높은 $k$개 토큰만 고려, 나머지는 0으로:
$$\mathcal{V}_k = \text{top-}k\{P(w)\}$$
$$\hat{P}(w) = \frac{P(w)}{\sum_{w' \in \mathcal{V}_k} P(w')} \text{ for } w \in \mathcal{V}_k, \text{ else } 0$$

## 24.5 Top-p (Nucleus) Sampling

확률 합이 $p$ 이상이 되는 최소 집합:
$$\mathcal{V}_p = \min \{S : \sum_{w \in S} P(w) \geq p\}$$

유동적: 분포가 평평하면 많은 토큰, 날카로우면 적은 토큰.


In [ ]:
# Top-k, Top-p 구현
def top_k_sampling(logits, k=5, tau=1.0):
    """Top-k Sample링."""
    p = softmax(logits.numpy(), tau=tau)
    # 상위 k개 인덱스
    top_idx = np.argsort(p)[-k:]
    mask = np.zeros_like(p)
    mask[top_idx] = p[top_idx]
    # Normalization
    mask = mask / mask.sum()
    return np.random.choice(len(p), p=mask)

def top_p_sampling(logits, p=0.9, tau=1.0):
    """Top-p (Nucleus) Sample링."""
    probs = softmax(logits.numpy(), tau=tau)
    # 확률 내림차순 정렬
    sorted_idx = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    # 누적
    cumsum = np.cumsum(sorted_probs)
    # p 이상이 되는 지점 찾기
    cutoff = np.searchsorted(cumsum, p) + 1
    # 상위 cutoff 개만 유지
    keep = sorted_idx[:cutoff]
    new_p = np.zeros_like(probs)
    new_p[keep] = probs[keep]
    new_p = new_p / new_p.sum()
    return np.random.choice(len(probs), p=new_p)

# 시각화: 각 전략별 선택 가능한 토큰
np.random.seed(42)
logits = np.random.randn(20) * 2
probs = softmax(logits, tau=1.0)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))

# Top-k 마스크 계산
top5_idx = np.argsort(probs)[-5:]
mask_topk = np.zeros(20)
mask_topk[top5_idx] = probs[top5_idx]
mask_topk = mask_topk / mask_topk.sum()

# Top-p 마스크 계산
sorted_idx = np.argsort(probs)[::-1]
sorted_probs = probs[sorted_idx]
cumsum = np.cumsum(sorted_probs)
cutoff = np.searchsorted(cumsum, 0.9) + 1
keep = sorted_idx[:cutoff]
mask_p = np.zeros(20); mask_p[keep] = probs[keep]
mask_p = mask_p / mask_p.sum()

# temperature
mask_t = softmax(logits, tau=0.5)

strategies = [
    ('Greedy', np.eye(20)[np.argmax(probs)]),
    ('Top-k=5', mask_topk),
    ('Top-p=0.9', mask_p),
    ('Temperature=0.5', mask_t),
]

for ax, (name, mask) in zip(axes, strategies):
    ax.bar(range(20), probs, alpha=0.3, color='gray', label='Original Distribution')
    ax.bar(range(20), mask, alpha=0.7, color='red', label='Line택 가능')
    ax.set_title(name)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch24_decoding_strategies.png', dpi=100, bbox_inches='tight')
plt.show()


## 24.6 Beam Search

$B$개의 부분 시퀀스(beam)를 유지, 각 스텝마다 확장:

점수: $\sum_t \log P(w_t | w_{<t})$

- 빔 폭 $B$ = 보통 4~10
- 품질은 좋지만 다양성 낮음
- LLM에서는 잘 안 쓰임 (느리고, 반복적)


In [ ]:
# Beam Search 간단 구현
def beam_search(initial_tokens, get_logits_fn, beam_width=4, max_len=20):
    """간단한 beam search."""
    beams = [(initial_tokens, 0.0)]  # (tokens, log_prob)
    for step in range(max_len):
        all_candidates = []
        for tokens, score in beams:
            logits = get_logits_fn(tokens)
            log_probs = F.log_softmax(logits[-1], dim=-1)
            # 상위 beam_width개
            topk = log_probs.topk(beam_width)
            for i in range(beam_width):
                new_tokens = tokens + [topk.indices[i].item()]
                new_score = score + topk.values[i].item()
                all_candidates.append((new_tokens, new_score))
        # 상위 beam_width개 Line택
        all_candidates.sort(key=lambda x: x[1], reverse=True)
        beams = all_candidates[:beam_width]
    return beams[0]  # 최고 Scores

# 가짜 로짓 Function
def fake_logits_fn(tokens):
    return torch.randn(len(tokens), 50)  # 어휘 50

result_tokens, result_score = beam_search([1, 2, 3], fake_logits_fn, beam_width=4, max_len=10)
print(f"Beam Search 결과: tokens={result_tokens}, score={result_score:.4f}")


## 24.7 Speculative Decoding

**아이디어**: 작은 초고속 모델(draft)이 먼저 $k$개 토큰 제안, 큰 모델(target)이 병렬 검증.

1. Draft 모델: $w_1^d, \ldots, w_k^d$ 생성 (빠름)
2. Target 모델: $P_T(w_1), \ldots, P_T(w_k)$ 병렬 계산 (한 번의 순전파)
3. 일치하면 수락, 불일치하면 해당 위치에서 재샘플링

효과: 2-3x 속도 향상 (Medusa, EAGLE 등)


## 24.8 Repetition Penalty

반복 방지: 이미 등장한 토큰의 로짓에 페널티.

$$\logit'(w) = \begin{cases} \logit(w) / p & \text{if } w \in \text{generated} \\ \logit(w) & \text{otherwise} \end{cases}$$

$p > 1$ (보통 1.1~1.3).

## 24.9 핵심 정리

| 전략 | 수식 | 특징 |
|---|---|---|
| Greedy | $\arg\max P$ | 결정적, 단조로움 |
| Temperature | $\sigma(z/\tau)$ | 분포 날카로움 조절 |
| Top-k | $\text{top-}k$ 토큰만 | 고정 k |
| Top-p | 누적합 $p$ | 유동적 |
| Beam | $B$개 시퀀스 | 품질 ↑, 다양성 ↓ |
| Speculative | draft + target | 2-3x 가속 |

## 연습문제

1. Greedy, Top-k=5, Top-p=0.9, Temperature=0.7로 같은 프롬프트에서 생성 비교하라.
2. Temperature 0.3, 0.7, 1.0, 1.5에 따른 엔트로피와 생성 다양성을 비교하라.
3. Beam Search의 빔 폭 1, 4, 8을 비교하라.
4. Repetition Penalty 1.0, 1.1, 1.3의 효과를 실험하라.
5. Speculative Decoding이 2-3x 빠른 이유를 수학적으로 설명하라.

> 해설: `solutions/ch24_solutions.ipynb`
